# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code beclow.

%load_ext dotenv
%dotenv 

In [2]:
import dask.dataframe as dd

c:\Users\Windows\anaconda3\envs\dsi_participant\lib\site-packages\dask\dataframe\__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
import os
from glob import glob
from pathlib import Path

PRICE_DATA = os.getenv('PRICE_DATA')
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)
dd_px = dd.read_parquet(parquet_files).set_index("Ticker")


c:\Users\Windows\anaconda3\envs\dsi_participant\lib\site-packages\dask\dataframe\core.py:5310: UserWarning: New index has same name as existing, this is a no-op.
  warnings.warn(


In [4]:
dd_px.compute()

Price,Date,Adj Close,Close,High,Low,Open,Volume,Year
Ticker,,,,,,,,
A,2000-01-03,NaN,43.382847,47.562963,40.596099,47.449986,4674353.0,2000
A,2000-01-04,NaN,40.068882,41.499914,39.014437,41.048007,4765083.0,2000
A,2000-01-05,NaN,37.583393,40.068869,36.340658,39.918233,5758642.0,2000
A,2000-01-06,NaN,36.152363,37.357444,35.022601,37.131491,2534434.0,2000
A,2000-01-07,NaN,39.165062,39.729943,35.549827,35.587484,2819626.0,2000
...,...,...,...,...,...,...,...,...
ZTS,2025-01-23,NaN,166.960007,167.300003,163.580002,166.389999,2318600.0,2025
ZTS,2025-01-24,NaN,168.610001,169.080002,165.949997,166.059998,2465600.0,2025
ZTS,2025-01-27,NaN,173.029999,173.479996,168.320007,168.320007,2404500.0,2025


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [5]:
# Write your code below.

dd_feat = dd_px.groupby('Ticker', group_keys=False).apply(
    lambda x: x.assign(Close_lag_1 = x['Close'].shift(1))
).assign(
    returns = lambda x: x['Close']/x['Close_lag_1'] - 1
).assign(
    hi_lo_range = lambda x: (x['High'] - x['Low']
))

C:\Users\Windows\AppData\Local\Temp\ipykernel_15164\1260615095.py:3: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_feat = (dd_px.groupby('Ticker', group_keys=False).apply(


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [6]:
pd_feat = dd_feat.compute()
pd_feat

Price,Date,Adj Close,Close,High,Low,Open,Volume,Year,Close_lag_1,returns,hi_lo_range
Ticker,,,,,,,,,,,
DOV,2008-01-02,NaN,21.197409,22.022972,21.192637,21.870267,1504321.0,2008,NaN,NaN,0.830335
DOV,2008-01-03,NaN,21.135380,21.412158,21.092432,21.264225,1533262.0,2008,21.197409,-0.002926,0.319726
DOV,2008-01-04,NaN,20.290731,21.059031,20.290731,20.977907,2272444.0,2008,21.135380,-0.039964,0.768299
DOV,2008-01-07,NaN,19.660818,20.519786,19.493797,20.500698,4208190.0,2008,20.290731,-0.031044,1.025989
DOV,2008-01-08,NaN,18.415310,19.756254,18.348501,19.713305,3965327.0,2008,19.660818,-0.063350,1.407753
...,...,...,...,...,...,...,...,...,...,...,...
CTLT,2020-12-24,NaN,105.309998,106.220001,104.660004,104.889999,277767.0,2020,104.860001,0.004291,1.559998
CTLT,2020-12-28,NaN,102.879997,106.550003,102.779999,106.139999,582003.0,2020,105.309998,-0.023075,3.770004
CTLT,2020-12-29,NaN,102.709999,103.644997,101.180000,103.370003,549151.0,2020,102.879997,-0.001652,2.464996


In [14]:
# Write your code below.
pd_feat['rolling_avg_return'] = pd_feat.groupby('Ticker')['returns'].transform(lambda x: x.rolling(10).mean())


In [13]:
pd_feat.sample(5)

Price,Date,Adj Close,Close,High,Low,Open,Volume,Year,Close_lag_1,returns,hi_lo_range,rolling_avg_return
Ticker,,,,,,,,,,,,
PSX,2003-11-06,NaN,NaN,NaN,NaN,NaN,NaN,2003,NaN,NaN,NaN,NaN
AVB,2022-12-20,NaN,148.748825,149.311674,147.346314,149.016403,642900.0,2022,149.680740,-0.006226,1.965360,-0.003380
ABT,2022-04-22,NaN,113.357208,116.711309,113.281408,116.559708,6710100.0,2022,116.891335,-0.030234,3.429901,-0.002479
GS,2013-08-07,NaN,131.549530,132.177297,130.422772,131.992189,2131000.0,2013,132.619934,-0.008071,1.754524,-0.000903
WBD,2022-09-27,NaN,11.320000,12.010000,11.220000,11.520000,22208200.0,2022,11.410000,-0.007888,0.790000,-0.012879


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?

I don't think so. I'd assume dask.DataFrame would have methods to group by and produce a rolling average. Pandas does it, for sure, but is probably the most efficient approach. 

+ Would it have been better to do it in Dask? Why?

Likely. The process of transforming a dask.DataFrame instance in to a pandas.DataFrame one, was lenghty and probably overwhel machine's memory.   


(1 pt)



## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.